# Создание базы данных для СУБД PostgreSQL из данных собранных парсеров в Telegram

Проект создан командой полной свэга SWAG

Руководитель команды - Новиков Александр Константинович

Участники команды - Крюков Алексей Алексеевич, Жыргалбек кызы Жасмин

Студенты 1-го курса МП "Социология публичной сферы и цифровая аналитика", трек "Цифровая аналитика социальных процессов"


Часть 1. Настройки подключение и загрузка необходимых библиотек

In [ ]:
import csv
import os
from typing import Dict, Tuple, Optional
import psycopg2

DB_NAME = "NAME HERE"
DB_USER = "postgres"
DB_PASSWORD = "PASSWORD"
DB_HOST = "127.0.0.1"
DB_PORT = "5432"

INPUT_DIR = "NAME HERE"

Часть 2. Вспомогательные функции

In [ ]:
def safe_int(value) -> Optional[int]:
    """Преобразует значение в целое число или возвращает None."""
    if value is None or value == "":
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def safe_text(value) -> Optional[str]:
    """Преобразует значение в строку без лишних пробелов или возвращает None."""
    if value is None:
        return None
    value = str(value).strip()
    return value if value != "" else None

def read_csv(filename: str):
    """Читает CSV-файл из папки INPUT_DIR."""
    path = os.path.join(INPUT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")

    with open(path, "r", encoding="utf-8") as file:
        return list(csv.DictReader(file))

Часть 3. Создание таблиц

In [ ]:
def recreate_tables(cur) -> None:
    """Полностью пересоздаёт таблицы в PostgreSQL с нуля."""
    cur.execute("""
        DROP TABLE IF EXISTS tg_reactions CASCADE;
        DROP TABLE IF EXISTS tg_comments CASCADE;
        DROP TABLE IF EXISTS tg_posts CASCADE;
        DROP TABLE IF EXISTS tg_chats CASCADE;
        DROP TABLE IF EXISTS tg_users CASCADE;
    """)

    cur.execute("""
        CREATE TABLE tg_users (
            user_id SERIAL PRIMARY KEY,
            tg_user_id BIGINT UNIQUE,
            display_name TEXT,
            username TEXT
        );
    """)

    cur.execute("""
        CREATE TABLE tg_chats (
            chat_id SERIAL PRIMARY KEY,
            tg_chat_id BIGINT UNIQUE,
            title TEXT,
            type TEXT,
            invite_link TEXT
        );
    """)

    cur.execute("""
        CREATE TABLE tg_posts (
            post_id SERIAL PRIMARY KEY,
            tg_message_id BIGINT NOT NULL,
            chat_id INTEGER NOT NULL REFERENCES tg_chats(chat_id),
            author_id INTEGER REFERENCES tg_users(user_id),
            published_at TIMESTAMP,
            text TEXT,
            view_count INTEGER,
            reaction_count INTEGER,
            comment_count INTEGER,
            CONSTRAINT uq_tg_posts_chat_message UNIQUE (chat_id, tg_message_id)
        );
    """)

    cur.execute("""
        CREATE TABLE tg_comments (
            comment_id SERIAL PRIMARY KEY,
            tg_message_id BIGINT NOT NULL,
            post_id INTEGER NOT NULL REFERENCES tg_posts(post_id),
            author_id INTEGER REFERENCES tg_users(user_id),
            published_at TIMESTAMP,
            text TEXT,
            reaction_count INTEGER,
            CONSTRAINT uq_tg_comments_post_message UNIQUE (post_id, tg_message_id)
        );
    """)

    cur.execute("""
        CREATE TABLE tg_reactions (
            reaction_id SERIAL PRIMARY KEY,
            entity_type TEXT NOT NULL,
            post_id INTEGER,
            comment_id INTEGER,
            reaction TEXT,
            reaction_count INTEGER
        );
    """)

Часть 4. Загрузка данных

In [ ]:
def load_users(cur, users_rows) -> Dict[int, int]:
    """Загружает пользователей и возвращает карту tg_user_id -> user_id."""
    user_map: Dict[int, int] = {}

    for row in users_rows:
        tg_user_id = safe_int(row.get("tg_user_id"))
        display_name = safe_text(row.get("display_name"))
        username = safe_text(row.get("username"))

        cur.execute(
            """
            INSERT INTO tg_users (tg_user_id, display_name, username)
            VALUES (%s, %s, %s)
            ON CONFLICT (tg_user_id) DO UPDATE
            SET display_name = EXCLUDED.display_name,
                username = EXCLUDED.username
            RETURNING user_id
            """,
            (tg_user_id, display_name, username)
        )

        user_id = cur.fetchone()[0]

        if tg_user_id is not None:
            user_map[tg_user_id] = user_id

    return user_map

def load_chats(cur, chats_rows) -> Dict[int, int]:
    """Загружает чаты и возвращает карту tg_chat_id -> chat_id."""
    chat_map: Dict[int, int] = {}

    for row in chats_rows:
        tg_chat_id = safe_int(row.get("tg_chat_id"))
        title = safe_text(row.get("title"))
        chat_type = safe_text(row.get("type"))
        invite_link = safe_text(row.get("invite_link"))

        cur.execute(
            """
            INSERT INTO tg_chats (tg_chat_id, title, type, invite_link)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (tg_chat_id) DO UPDATE
            SET title = EXCLUDED.title,
                type = EXCLUDED.type,
                invite_link = EXCLUDED.invite_link
            RETURNING chat_id
            """,
            (tg_chat_id, title, chat_type, invite_link)
        )

        chat_id = cur.fetchone()[0]

        if tg_chat_id is not None:
            chat_map[tg_chat_id] = chat_id

    return chat_map

def load_posts(cur, posts_rows, user_map: Dict[int, int], chat_map: Dict[int, int]) -> Dict[Tuple[int, int], int]:
    """Загружает посты и возвращает карту (tg_chat_id, tg_message_id) -> post_id."""
    post_map: Dict[Tuple[int, int], int] = {}

    for row in posts_rows:
        tg_message_id = safe_int(row.get("tg_message_id"))
        tg_chat_id = safe_int(row.get("tg_chat_id"))
        author_tg_user_id = safe_int(row.get("author_tg_user_id"))

        if tg_message_id is None or tg_chat_id is None:
            continue

        chat_id = chat_map.get(tg_chat_id)
        author_id = user_map.get(author_tg_user_id) if author_tg_user_id is not None else None

        if chat_id is None:
            continue

        cur.execute(
            """
            INSERT INTO tg_posts (
                tg_message_id,
                chat_id,
                author_id,
                published_at,
                text,
                view_count,
                reaction_count,
                comment_count
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (chat_id, tg_message_id) DO UPDATE
            SET author_id = EXCLUDED.author_id,
                published_at = EXCLUDED.published_at,
                text = EXCLUDED.text,
                view_count = EXCLUDED.view_count,
                reaction_count = EXCLUDED.reaction_count,
                comment_count = EXCLUDED.comment_count
            RETURNING post_id
            """,
            (
                tg_message_id,
                chat_id,
                author_id,
                safe_text(row.get("published_at")),
                row.get("text") or "",
                safe_int(row.get("view_count")),
                safe_int(row.get("reaction_count")) or 0,
                safe_int(row.get("comment_count")) or 0
            )
        )

        post_id = cur.fetchone()[0]
        post_map[(tg_chat_id, tg_message_id)] = post_id

    return post_map

def load_comments(
    cur,
    comments_rows,
    user_map: Dict[int, int],
    post_map: Dict[Tuple[int, int], int]
) -> Dict[Tuple[int, int], int]:
    """
    Загружает комментарии.
    Возвращает карту:
    (tg_chat_id, tg_message_id) -> comment_id
    """
    comment_map: Dict[Tuple[int, int], int] = {}

    for row in comments_rows:
        comment_tg_message_id = safe_int(row.get("tg_message_id"))
        comment_tg_chat_id = safe_int(row.get("tg_chat_id"))
        post_tg_message_id = safe_int(row.get("post_tg_message_id"))
        author_tg_user_id = safe_int(row.get("author_tg_user_id"))

        if comment_tg_message_id is None or comment_tg_chat_id is None or post_tg_message_id is None:
            continue

        post_id = post_map.get((comment_tg_chat_id, post_tg_message_id))
        if post_id is None:
            continue

        author_id = user_map.get(author_tg_user_id) if author_tg_user_id is not None else None

        cur.execute(
            """
            INSERT INTO tg_comments (
                tg_message_id,
                post_id,
                author_id,
                published_at,
                text,
                reaction_count
            )
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (post_id, tg_message_id) DO UPDATE
            SET author_id = EXCLUDED.author_id,
                published_at = EXCLUDED.published_at,
                text = EXCLUDED.text,
                reaction_count = EXCLUDED.reaction_count
            RETURNING comment_id
            """,
            (
                comment_tg_message_id,
                post_id,
                author_id,
                safe_text(row.get("published_at")),
                row.get("text") or "",
                safe_int(row.get("reaction_count")) or 0
            )
        )

        comment_id = cur.fetchone()[0]
        comment_map[(comment_tg_chat_id, comment_tg_message_id)] = comment_id

    return comment_map

def load_reactions(
    cur,
    reactions_rows,
    post_map: Dict[Tuple[int, int], int],
    comment_map: Dict[Tuple[int, int], int]
) -> None:
    """Загружает реакции."""
    for row in reactions_rows:
        entity_type = safe_text(row.get("entity_type"))
        owner_tg_chat_id = safe_int(row.get("owner_tg_chat_id"))
        owner_tg_message_id = safe_int(row.get("owner_tg_message_id"))
        reaction = safe_text(row.get("reaction"))
        reaction_count = safe_int(row.get("reaction_count")) or 0

        if entity_type not in {"post", "comment"}:
            continue

        if owner_tg_chat_id is None or owner_tg_message_id is None:
            continue

        post_id = None
        comment_id = None

        if entity_type == "post":
            post_id = post_map.get((owner_tg_chat_id, owner_tg_message_id))
            if post_id is None:
                continue

        if entity_type == "comment":
            comment_id = comment_map.get((owner_tg_chat_id, owner_tg_message_id))
            if comment_id is None:
                continue

        cur.execute(
            """
            INSERT INTO tg_reactions (
                entity_type,
                post_id,
                comment_id,
                reaction,
                reaction_count
            )
            VALUES (%s, %s, %s, %s, %s)
            """,
            (entity_type, post_id, comment_id, reaction, reaction_count)
        )

Часть 5. Проверка результата

In [ ]:
def print_counts(cur) -> None:
    """Печатает количество строк в таблицах."""
    cur.execute("SELECT COUNT(*) FROM tg_users")
    print("Пользователи:", cur.fetchone()[0])

    cur.execute("SELECT COUNT(*) FROM tg_chats")
    print("Чаты и каналы:", cur.fetchone()[0])

    cur.execute("SELECT COUNT(*) FROM tg_posts")
    print("Посты:", cur.fetchone()[0])

    cur.execute("SELECT COUNT(*) FROM tg_comments")
    print("Комментарии:", cur.fetchone()[0])

    cur.execute("SELECT COUNT(*) FROM tg_reactions")
    print("Реакции:", cur.fetchone()[0])

Часть 6. Основной запуск программы

In [ ]:
def main():
    if not os.path.exists(INPUT_DIR):
        raise FileNotFoundError(f"Папка не найдена: {INPUT_DIR}")

    print("Читаем CSV-файлы...")
    users_rows = read_csv("tg_users.csv")
    chats_rows = read_csv("tg_chats.csv")
    posts_rows = read_csv("tg_posts.csv")
    comments_rows = read_csv("tg_comments.csv")
    reactions_rows = read_csv("tg_reactions.csv")
    print("CSV-файлы прочитаны.")

    print("Подключаемся к PostgreSQL...")
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )

    conn.autocommit = False
    cur = conn.cursor()

    try:
        print("Полностью пересоздаём таблицы...")
        recreate_tables(cur)
        print("Таблицы пересозданы.")

        print("Загружаем пользователей...")
        user_map = load_users(cur, users_rows)
        print(f"Пользователи загружены: {len(user_map)}")

        print("Загружаем чаты и каналы...")
        chat_map = load_chats(cur, chats_rows)
        print(f"Чаты и каналы загружены: {len(chat_map)}")

        print("Загружаем посты...")
        post_map = load_posts(cur, posts_rows, user_map, chat_map)
        print(f"Посты загружены: {len(post_map)}")

        print("Загружаем комментарии...")
        comment_map = load_comments(cur, comments_rows, user_map, post_map)
        print(f"Комментарии загружены: {len(comment_map)}")

        print("Загружаем реакции...")
        load_reactions(cur, reactions_rows, post_map, comment_map)
        print("Реакции загружены.")

        conn.commit()

        print("Загрузка завершена.")
        print_counts(cur)

    except Exception as error:
        conn.rollback()
        print("Произошла ошибка. Транзакция отменена.")
        raise error

    finally:
        cur.close()
        conn.close()
        print("Соединение с PostgreSQL закрыто.")

if __name__ == "__main__":
    main()